# Exercice 3 — De la donnée brute au rapport d'analyse — CORRECTION

**Difficulté :** Avancé  
**Durée estimée :** 60 min  

---

## Contexte

Le responsable énergie du groupe demande une analyse complète des consommations énergétiques pour préparer un plan de réduction des coûts. Il dispose du fichier `consommation_energie.csv` qui couvre deux années de relevés quotidiens pour plusieurs bâtiments.

**Objectif :** Produire une analyse complète avec exploration, nettoyage, KPIs, visualisations et recommandations exportables.

---

## Tâche 1 — Explorer le dataset complet

In [ ]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import numpy as np

# Chargement du fichier
df = pd.read_csv('../data/consommation_energie.csv')

print(f"Dimensions : {df.shape[0]} lignes × {df.shape[1]} colonnes")
print()
df.head(10)

In [ ]:
# Résumé technique complet
df.info()

In [ ]:
# Statistiques descriptives
df.describe().round(2)

In [ ]:
# Valeurs uniques par colonne catégorielle
for col in ['batiment', 'zone', 'type_energie']:
    print(f"\n{col} :")
    print(df[col].value_counts())

In [ ]:
# Diagnostic des anomalies
print("Valeurs manquantes :")
print(df.isnull().sum())
print()
print(f"Doublons : {df.duplicated().sum()}")
print()
print(f"Valeurs kwh négatives : {(df['kwh'] < 0).sum()}")
print(f"Valeur kwh maximale : {df['kwh'].max():.2f} (à vérifier)")

**Explication :** L'exploration suit toujours le même ordre : dimensions → types → statistiques → valeurs uniques → anomalies. On identifie ici trois problèmes à traiter : des doublons (15), des kwh négatifs (10) et des valeurs manquantes dans la colonne `kwh` (20). La valeur max de 6734 kWh mérite vérification mais n'est pas nécessairement aberrante pour une usine.

## Tâche 2 — Nettoyer les données

In [ ]:
taille_initiale = len(df)
print(f"Lignes initiales : {taille_initiale}")

# Étape 1 : supprimer les doublons exacts
df = df.drop_duplicates()
print(f"Après suppression doublons : {len(df)} lignes (-{taille_initiale - len(df)})")

# Étape 2 : supprimer les lignes avec kwh négatifs (mesures impossibles)
avant = len(df)
df = df[df['kwh'] >= 0]
print(f"Après suppression kwh négatifs : {len(df)} lignes (-{avant - len(df)})")

# Étape 3 : supprimer les lignes avec kwh manquant
# (pas de valeur = pas de relevé fiable, on ne peut pas imputer une consommation)
avant = len(df)
df = df.dropna(subset=['kwh'])
print(f"Après suppression kwh manquants : {len(df)} lignes (-{avant - len(df)})")

# Étape 4 : conversion de la date
df['date'] = pd.to_datetime(df['date'])
df['mois'] = df['date'].dt.to_period('M')   # colonne auxiliaire pour les agrégations mensuelles
df['annee'] = df['date'].dt.year
df['mois_num'] = df['date'].dt.month

print(f"\nDataset nettoyé : {len(df)} lignes")
print(f"Période couverte : {df['date'].min().date()} → {df['date'].max().date()}")

**Explication :** Chaque décision de nettoyage a une justification métier :

- **Doublons** : une même mesure ne doit compter qu'une fois. `drop_duplicates()` compare l'ensemble des colonnes par défaut.
- **Valeurs négatives** : une consommation en kWh est physiquement positive. Ces valeurs sont des erreurs de saisie ou de transmission.
- **Valeurs manquantes** : on préfère ignorer une mesure inconnue plutôt que de l'imputer (inventer une valeur), ce qui introduirait un biais dans les totaux.
- **Colonnes auxiliaires** : `mois` et `annee` facilitent les agrégations sans avoir à répéter les transformations de date à chaque étape.

## Tâche 3 — Calculer les KPIs

In [ ]:
# KPI 1 : consommation totale
total_kwh = df['kwh'].sum()
print(f"Consommation totale : {total_kwh:,.0f} kWh")
print()

# KPI 2 : consommation moyenne journalière par bâtiment
nb_jours = (df['date'].max() - df['date'].min()).days + 1
moy_par_batiment = (
    df.groupby('batiment')['kwh']
    .sum()
    .div(nb_jours)   # diviser par le nombre de jours
    .round(1)
    .sort_values(ascending=False)
)
print("Consommation moyenne journalière par bâtiment (kWh/jour) :")
print(moy_par_batiment)
print()

# KPI 3 : ratio gaz / électricité
par_type = df.groupby('type_energie')['kwh'].sum()
ratio = (par_type / par_type.sum() * 100).round(1)
print("Répartition par vecteur énergétique :")
for energie, pct in ratio.items():
    print(f"  {energie} : {pct} % ({par_type[energie]:,.0f} kWh)")
print()

# KPI 4 : variation saisonnière
ete = df[df['mois_num'].isin([6, 7, 8])]['kwh'].mean()
hiver = df[df['mois_num'].isin([12, 1, 2])]['kwh'].mean()
variation = ((hiver - ete) / ete * 100).round(1)
print("Variation saisonnière (consommation quotidienne moyenne) :")
print(f"  Été (juin-août)          : {ete:.1f} kWh/relevé")
print(f"  Hiver (déc-jan-fév)      : {hiver:.1f} kWh/relevé")
print(f"  Variation hiver vs été   : +{variation} %" if variation > 0 else f"  Variation hiver vs été : {variation} %")

**Explication :** Les KPIs doivent répondre directement aux questions métier. On distingue :
- La consommation **totale** pour dimensionner l'enjeu
- La consommation **par unité de temps** pour comparer les bâtiments équitablement
- La **répartition** entre vecteurs pour orienter les leviers d'action (gaz vs électricité n'ont pas le même coût ni le même impact carbone)
- La **saisonnalité** pour identifier les postes pilotables (chauffage, climatisation)

## Tâche 4 — Graphique par bâtiment

In [ ]:
# Agréger la consommation totale par bâtiment et type d'énergie
conso_bat_type = (
    df.groupby(['batiment', 'type_energie'])['kwh']
    .sum()
    .reset_index()
)
conso_bat_type['kwh_milliers'] = (conso_bat_type['kwh'] / 1000).round(1)

fig4 = px.bar(
    conso_bat_type,
    x='batiment',
    y='kwh_milliers',
    color='type_energie',
    barmode='group',                      # barres côte à côte (vs 'stack' pour empilées)
    title='Consommation totale par bâtiment et par type d\'énergie (2023-2024)',
    labels={
        'batiment': 'Bâtiment',
        'kwh_milliers': 'Consommation (MWh)',
        'type_energie': "Type d'énergie"
    },
    color_discrete_map={
        'electricite': '#2196F3',
        'gaz': '#FF9800'
    },
    text='kwh_milliers'
)

fig4.update_traces(texttemplate='%{text:.0f}', textposition='outside')
fig4.update_layout(
    plot_bgcolor='white',
    yaxis_title='Consommation (MWh)',
    legend_title_text="Type d'énergie"
)

fig4.show()

**Interprétation :** Le graphique en barres groupées permet de comparer simultanément les bâtiments entre eux et la répartition gaz/électricité au sein de chaque bâtiment. Les usines de production consomment logiquement plus que les bureaux. Une prédominance du gaz dans certains bâtiments indique une dépendance aux chaudières industrielles — un levier d'optimisation différent de l'électricité.

## Tâche 5 — Graphique temporel avec tendance

In [ ]:
# Agréger par mois et type d'énergie
conso_mensuelle = (
    df.groupby([df['date'].dt.to_period('M'), 'type_energie'])['kwh']
    .sum()
    .reset_index()
)
# Convertir Period en timestamp pour plotly (qui ne gère pas nativement Period)
conso_mensuelle['date'] = conso_mensuelle['date'].dt.to_timestamp()
conso_mensuelle['kwh_milliers'] = (conso_mensuelle['kwh'] / 1000).round(1)

fig5 = px.line(
    conso_mensuelle,
    x='date',
    y='kwh_milliers',
    color='type_energie',
    title='Évolution mensuelle de la consommation par type d\'énergie (2023-2024)',
    labels={
        'date': 'Mois',
        'kwh_milliers': 'Consommation (MWh)',
        'type_energie': "Type d'énergie"
    },
    markers=True,                         # points visibles sur la courbe
    color_discrete_map={
        'electricite': '#2196F3',
        'gaz': '#FF9800'
    }
)

fig5.update_layout(
    plot_bgcolor='white',
    xaxis=dict(tickformat='%b %Y'),       # format des dates sur l'axe
    legend_title_text="Type d'énergie"
)

fig5.show()

**Interprétation :** La visualisation temporelle révèle la saisonnalité : on s'attend à des pics de consommation de gaz en hiver (chauffage) et d'électricité en été (climatisation). Si les deux courbes progressent en parallèle sur la période, cela indique une croissance globale de la consommation. Une divergence entre 2023 et 2024 pour le même mois indiquerait un changement structurel (nouvel équipement, nouvelles activités).

**Explication technique :** `.dt.to_period('M')` convertit les dates en périodes mensuelles pour l'agrégation, puis `.dt.to_timestamp()` les reconvertit en dates pour la compatibilité avec plotly.

## Tâche 6 — Heatmap zones × mois

In [ ]:
# Noms des mois pour l'axe X
noms_mois = {
    1: 'Jan', 2: 'Fév', 3: 'Mar', 4: 'Avr',
    5: 'Mai', 6: 'Juin', 7: 'Juil', 8: 'Août',
    9: 'Sep', 10: 'Oct', 11: 'Nov', 12: 'Déc'
}

# Créer le tableau croisé dynamique : zones en lignes, mois en colonnes
pivot = df.pivot_table(
    values='kwh',
    index='zone',
    columns='mois_num',
    aggfunc='mean'       # consommation moyenne mensuelle
).round(0)

# Renommer les colonnes avec les noms de mois
pivot.columns = [noms_mois[m] for m in pivot.columns]

fig6 = px.imshow(
    pivot,
    title='Consommation moyenne par zone et par mois (kWh)',
    labels=dict(x='Mois', y='Zone', color='kWh moyen'),
    color_continuous_scale='RdYlGn_r',   # rouge = forte consommation, vert = faible
    aspect='auto',
    text_auto=True                        # afficher les valeurs dans chaque cellule
)

fig6.update_layout(
    xaxis_title='Mois',
    yaxis_title='Zone'
)

fig6.show()

**Interprétation :** La heatmap est particulièrement efficace pour identifier des patterns croisés. Les cellules les plus rouges indiquent les combinaisons zone × mois à fort impact. On s'attend à voir la zone Production consommer davantage en période d'activité soutenue, et la Climatisation pic en juillet-août.

**Explication technique :** `pivot_table()` est l'équivalent pandas d'un tableau croisé dynamique Excel. `aggfunc='mean'` calcule la moyenne (on pourrait aussi utiliser `'sum'` pour le total). `px.imshow()` est la fonction plotly express dédiée aux matrices de valeurs.

## Tâche 7 — Identifier les 3 postes de consommation principaux

In [ ]:
# Agréger par bâtiment + zone (= un poste de consommation)
postes = (
    df.groupby(['batiment', 'zone'])['kwh']
    .sum()
    .reset_index()
    .sort_values('kwh', ascending=False)
)

total = postes['kwh'].sum()
postes['part_pct'] = (postes['kwh'] / total * 100).round(1)
postes['kwh'] = postes['kwh'].round(0).astype(int)

print("Top 10 des postes de consommation :")
print(postes.head(10).to_string(index=False))
print()

top3 = postes.head(3)
print("Les 3 postes principaux représentent :")
print(f"{top3['part_pct'].sum():.1f} % de la consommation totale")
print(f"soit {top3['kwh'].sum():,} kWh")

**Explication :** La combinaison `batiment + zone` définit un poste de consommation précis et actionnable. Cette granularité est nécessaire pour formuler des recommandations concrètes : savoir que "la Production consomme beaucoup" est moins utile que savoir que "la Production de l'Usine_A représente 18 % du total".

## Tâche 8 — Recommandations

In [ ]:
# Données pour étayer les recommandations
variation_saisonniere = (
    df[df['mois_num'].isin([12, 1, 2])]['kwh'].mean() -
    df[df['mois_num'].isin([6, 7, 8])]['kwh'].mean()
) / df[df['mois_num'].isin([6, 7, 8])]['kwh'].mean() * 100

print(f"Variation saisonnière hiver/été : +{variation_saisonniere:.0f} %")
print(f"Top 3 postes : {top3[['batiment', 'zone', 'part_pct']].to_string(index=False)}")

**Recommandation 1 — Optimiser la consommation du poste de production principal**

Le premier poste de consommation (bâtiment × zone Production) représente une part disproportionnée du total. Un audit énergétique ciblé sur ce poste, suivi d'un programme de réglage des équipements (température de consigne, séquençage des démarrages), pourrait générer 5 à 10 % d'économies sur ce périmètre sans investissement majeur.

**Recommandation 2 — Réduire la consommation hors heures de production**

La variation saisonnière identifiée indique un fort impact du chauffage/climatisation. La mise en place d'une gestion programmée (GTB — Gestion Technique du Bâtiment) avec des plages de relâche la nuit et le week-end peut réduire ce poste de 15 à 20 %.

**Recommandation 3 — Rééquilibrer le mix gaz / électricité**

Le ratio gaz/électricité observé peut représenter un risque de coût selon l'évolution des prix de marché. Étudier la substitution partielle du gaz par des solutions électriques (pompes à chaleur, résistances haute performance) sur les postes à fort volume de gaz permettrait de sécuriser le budget énergie à moyen terme.

## Tâche 9 — Exporter les résultats

In [ ]:
import os

# Créer un dossier de sortie
output_dir = '../output'
os.makedirs(output_dir, exist_ok=True)

# Export 1 : graphique temporel en HTML autonome
fig5.write_html(
    f'{output_dir}/consommation_mensuelle.html',
    include_plotlyjs='cdn'   # lien CDN = fichier léger (pas d'intégration du JS)
)
print("Graphique exporté : output/consommation_mensuelle.html")

# Export 2 : tableau récapitulatif CSV
rapport = (
    df.groupby(['batiment', 'zone', 'type_energie'])['kwh']
    .sum()
    .reset_index()
)
rapport.columns = ['batiment', 'zone', 'type_energie', 'consommation_totale_kwh']
rapport['consommation_totale_kwh'] = rapport['consommation_totale_kwh'].round(0).astype(int)
rapport['part_du_total_pct'] = (
    rapport['consommation_totale_kwh'] / rapport['consommation_totale_kwh'].sum() * 100
).round(2)
rapport = rapport.sort_values('consommation_totale_kwh', ascending=False)

rapport.to_csv(f'{output_dir}/rapport_consommation.csv', index=False)
print("Rapport exporté : output/rapport_consommation.csv")
print()
print("Aperçu du rapport :")
rapport.head(10)

**Explication :** `write_html()` génère un fichier HTML autonome qui contient le graphique interactif plotly. L'option `include_plotlyjs='cdn'` maintient le fichier léger (quelques Ko) en chargeant la bibliothèque plotly depuis un serveur externe — ce fichier nécessite une connexion internet pour s'afficher. Pour un rapport hors-ligne, utiliser `include_plotlyjs=True` (le fichier fera ~3 Mo).

## Tâche 10 — Bonus : estimation d'économie potentielle

In [ ]:
# Consommation totale de l'ensemble du dataset
consommation_totale = df['kwh'].sum()

# Consommation des 3 postes principaux
conso_top3 = top3['kwh'].sum()

# Économie si réduction de 10 % sur ces 3 postes
taux_reduction = 0.10
economie_kwh = conso_top3 * taux_reduction
economie_pct_total = economie_kwh / consommation_totale * 100

# Hypothèse tarifaire (prix moyen indicatif)
prix_kwh_electricite = 0.18   # €/kWh
prix_kwh_gaz = 0.07           # €/kWh
prix_moyen = (prix_kwh_electricite + prix_kwh_gaz) / 2

economie_euros = economie_kwh * prix_moyen

print("=" * 50)
print("ESTIMATION D'ÉCONOMIE POTENTIELLE")
print("=" * 50)
print(f"Consommation totale         : {consommation_totale:>12,.0f} kWh")
print(f"Consommation Top 3 postes   : {conso_top3:>12,.0f} kWh")
print(f"Réduction appliquée         : {taux_reduction:.0%}")
print(f"Économie estimée            : {economie_kwh:>12,.0f} kWh")
print(f"Soit                        : {economie_pct_total:>11.1f} % du total")
print(f"Économie financière estimée : {economie_euros:>11,.0f} €/an")
print()
print("(basé sur un prix moyen indicatif de 0.125 €/kWh)")

**Explication :** Ce calcul illustre la méthode Pareto appliquée à l'énergie : concentrer les efforts sur les 3 postes les plus consommateurs pour obtenir le maximum d'impact avec le minimum d'actions. La valorisation financière (même approximative) est nécessaire pour convaincre la direction d'allouer un budget à ces optimisations.

---

## Synthèse de l'analyse

| Indicateur | Valeur |
|---|---|
| Période couverte | 2023-2024 (2 ans) |
| Bâtiments analysés | 4 |
| Lignes nettoyées | 45 (doublons + anomalies) |
| Consommation totale | voir KPI 1 |
| Top 3 postes — part du total | voir Tâche 7 |
| Économie potentielle (−10 % top 3) | voir Tâche 10 |

Les livrables exportés (HTML + CSV) sont disponibles dans le dossier `output/`.